In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.appName("testing").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/08 04:49:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/08 04:50:06 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [3]:
# read csv
accounts = spark.read.option('header','true').csv('./data/accounts.csv')
accounts.printSchema()

root
 |-- account_number: string (nullable = true)
 |-- aba: string (nullable = true)
 |-- bic: string (nullable = true)
 |-- opened: string (nullable = true)
 |-- balance: string (nullable = true)



In [4]:
# read json
clients = spark.read.json('./data/clients.json')
clients.printSchema()

root
 |-- account_number: string (nullable = true)
 |-- address: string (nullable = true)
 |-- email: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)



In [6]:
# read parquet
transactions = spark.read.option('header', True).parquet('./data/transactions.parquet')
transactions.printSchema()

root
 |-- account_number: string (nullable = true)
 |-- amount: long (nullable = true)
 |-- datetime: date (nullable = true)



In [10]:
# group transactions by account number
account_transactions = transactions.groupby('account_number').sum()
account_transactions.show(5)

+------------------+-----------+
|    account_number|sum(amount)|
+------------------+-----------+
|FBXK78425844480007|     -99434|
|XJIU55438863095422|      77947|
|XBYT37304125118047|      65101|
|UQSE17000937342665|     118473|
|KWOU43650129218895|     -35411|
+------------------+-----------+
only showing top 5 rows



In [12]:
with_sum = account_transactions.join(accounts, 'account_number', 'inner')
with_sum.printSchema()

root
 |-- account_number: string (nullable = true)
 |-- sum(amount): long (nullable = true)
 |-- aba: string (nullable = true)
 |-- bic: string (nullable = true)
 |-- opened: string (nullable = true)
 |-- balance: string (nullable = true)



In [20]:
accounts = with_sum.withColumn('new_balance', sum([with_sum['balance'], with_sum["sum(amount)"]]))
neg_balance = accounts.filter(accounts['new_balance'] < 0)
neg_balance.show(5)

+------------------+-----------+---------+-----------+----------+-------+-----------+
|    account_number|sum(amount)|      aba|        bic|    opened|balance|new_balance|
+------------------+-----------+---------+-----------+----------+-------+-----------+
|JMTP45763117901514|     -85521|108480276|   AKJGGBSL|2000-11-08|  57948|   -27573.0|
|RBUE05237750254383|    -138553|054916920|   EXUPGBRO|2010-09-07|  35094|  -103459.0|
|RJMY57096756148587|    -102019|098748471|   SPIOGBOT|2005-05-11|  43690|   -58329.0|
|ZYMB62177146259441|     -81891|115359235|   XWITGB3O|2014-01-21|  26460|   -55431.0|
|LRTH65732611614073|    -154066|108298031|ZWIOGB1G1I8|2013-05-27|  50235|  -103831.0|
+------------------+-----------+---------+-----------+----------+-------+-----------+
only showing top 5 rows



In [22]:
clients = clients.join(neg_balance, 'account_number', 'inner')
clients_disp = clients.select(['first_name', 'last_name', 'new_balance'])
clients_disp.show(5)

+----------+---------+-----------+
|first_name|last_name|new_balance|
+----------+---------+-----------+
|    Meagan| Sandoval|   -27573.0|
|  Michelle|   Knight|  -103459.0|
|      Paul|   Massey|   -58329.0|
|  Michelle|    Perez|   -55431.0|
|     David|    Green|  -103831.0|
+----------+---------+-----------+
only showing top 5 rows

